In [0]:
%sql
-- COMMAND ----------

DESCRIBE DETAIL aml_engine.aml_poc.silver_alerts;

In [0]:
%sql
-- COMMAND ----------

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT alert_id) AS unique_alert_ids,
    COUNT(DISTINCT tx_id) AS unique_tx_ids,

    SUM(
        CASE
            WHEN alert_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_alert_ids,

    SUM(
        CASE
            WHEN tx_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_tx_ids,

    SUM(
        CASE
            WHEN tx_amount <= 0 THEN 1
            ELSE 0
        END
    ) AS invalid_amounts

FROM aml_engine.aml_poc.silver_alerts;

In [0]:
%sql
-- COMMAND ----------

SELECT
    alert_id,
    tx_id,
    COUNT(*) AS record_count
FROM aml_engine.aml_poc.silver_alerts
GROUP BY
    alert_id,
    tx_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- COMMAND ----------

DESCRIBE DETAIL aml_engine.aml_poc.silver_transactions;

In [0]:
%sql
-- COMMAND ----------

SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT tx_id) AS unique_tx_ids,

    SUM(
        CASE
            WHEN tx_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_tx_ids,

    SUM(
        CASE
            WHEN sender_account_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_sender_ids,

    SUM(
        CASE
            WHEN receiver_account_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_receiver_ids,

    SUM(
        CASE
            WHEN tx_amount IS NULL THEN 1
            ELSE 0
        END
    ) AS null_amounts,

    SUM(
        CASE
            WHEN tx_amount <= 0 THEN 1
            ELSE 0
        END
    ) AS invalid_amounts

FROM aml_engine.aml_poc.silver_transactions;

In [0]:
%sql
-- COMMAND ----------

SELECT
    tx_id,
    COUNT(*) AS record_count
FROM aml_engine.aml_poc.silver_transactions
GROUP BY tx_id
HAVING COUNT(*) > 1;

In [0]:
%sql
-- COMMAND ----------

SELECT COUNT(*) AS rescued_records
FROM aml_engine.aml_poc.bronze_transactions
WHERE _rescued_data IS NOT NULL;

In [0]:
%sql
SELECT
    is_fraud,
    COUNT(*) AS transaction_count,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        4
    ) AS percentage
FROM aml_engine.aml_poc.silver_transactions
GROUP BY is_fraud
ORDER BY is_fraud;

In [0]:
%sql
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT tx_id) AS unique_transactions
FROM aml_engine.aml_poc.silver_transactions;

In [0]:
%sql
SELECT
    COUNT(*) AS alert_rows,
    COUNT(DISTINCT tx_id) AS alerted_transactions
FROM aml_engine.aml_poc.silver_alerts;

In [0]:
%sql
select * from aml_engine.aml_poc.silver_transactions where tx_id = 1077827;

In [0]:
%sql
select * from aml_engine.aml_poc.silver_alerts where tx_id = 1077827;

In [0]:
%sql
select * from aml_engine.aml_poc.ml_training_data where tx_id = 1077827;

In [0]:
%sql

SELECT *
FROM aml_engine.aml_poc.silver_transactions
WHERE sender_account_id IN (8680, 8959)
   OR receiver_account_id IN (8680, 8959)
ORDER BY event_time
LIMIT 200;

In [0]:
tx.filter(
    (col("event_time") >= 150) &
    (col("event_time") <= 170)
).filter(
    (col("sender_account_id").isin([8680, 8959])) |
    (col("receiver_account_id").isin([8680, 8959]))
).orderBy("event_time", "tx_id").show(200, truncate=False)

In [0]:
from pyspark.sql.functions import col

tx = spark.table("aml_engine.aml_poc.silver_transactions")

tx.filter(
    (col("sender_account_id").isin([8680, 8959])) |
    (col("receiver_account_id").isin([8680, 8959]))
).orderBy("event_time").show(200, truncate=False)

In [0]:
tx.filter(
    (col("event_time") >= 100) &
    (col("event_time") <= 220)
).filter(
    (col("sender_account_id").isin([8680, 8959])) |
    (col("receiver_account_id").isin([8680, 8959]))
).orderBy("event_time").show(200, truncate=False)

In [0]:
tx.filter(
    (col("event_time") >= 158) &
    (col("event_time") <= 170)
).filter(
    (col("sender_account_id").isin([8680, 8959])) |
    (col("receiver_account_id").isin([8680, 8959]))
).orderBy("event_time", "tx_id").show(200, truncate=False)

In [0]:
tx.filter(
    (col("sender_account_id") == 9084) &
    (col("receiver_account_id") == 5687)
).orderBy("event_time", "tx_id").show(100, truncate=False)

In [0]:
tx.filter(
    (col("event_time") >= 140) &
    (col("event_time") <= 180)
).filter(
    col("sender_account_id").isin([8680, 8959, 9084, 5687]) |
    col("receiver_account_id").isin([8680, 8959, 9084, 5687])
).select(
    "tx_id",
    "sender_account_id",
    "receiver_account_id",
    "tx_type",
    "tx_amount",
    "event_time",
    "is_fraud",
    "alert_id"
).orderBy(
    "event_time",
    "tx_id"
).show(500, truncate=False)